# 13장 실습. 교사 분포

파인튜닝은 정답 하나가 아니라 **분포**를 배운다. 학습 데이터 한 줄마다 「이 선택지가 맞을 확률」을 선택지 수만큼 적어 둔다. 그 분포를 주는 쪽이 교사다. 이 노트북은 교사 다섯을 만들고, 품질을 재고, 14장이 읽을 학습 파일을 쓴다.

| 교사 | 분포 | 비용 |
|---|---|---|
| hard | 정답 하나에 몰아준다 (라벨 스무딩 0.1) | 없음 |
| human | KLUE 주석자 표의 비율 (NSMC 는 표가 없어 hard 로 대신한다) | 없음 |
| jev | Jev 가 학습 데이터에 낸 확률 | 입력 토큰 과금 (약 0.11달러) |
| llm | 로컬 LLM 의 첫 토큰 확률 (AnyJev 방식, 순환 순열) | GPU 시간 |
| mix | 0.5 · jev + 0.5 · human | jev 와 같다 |

| 절 | 내용 |
|---|---|
| 13.1 | 학습 데이터 뽑기 — train 분할에서 과제마다 1,500개 |
| 13.2 | 사람 표 분포 — 얼마나 갈리는가 |
| 13.3 | Jev 교사 |
| 13.4 | 로컬 LLM 교사 |
| 13.5 | 두 교사가 갈린 항목 |
| 13.6 | 교사 품질 — 평가셋에서 교사끼리, 교사와 사람을 견준다 |
| 13.7 | 학습 파일 쓰기와 사전 점검 (`laya.train.dry_run`) |

Jev 출력을 학습 데이터로 쓰기 전에 TypeSafe 이용 약관이 그것을 허용하는지 확인한다(9장). 허용되지 않으면 jev·mix 교사를 빼고 llm·human 교사로 진행한다.

13.6 은 8장의 Jev 기록과 12장의 AnyJev 기록(`results/eval_v1/`)을 읽는다.

In [1]:
import os, json, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
import koreanize_matplotlib  # noqa: F401
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter, Retry

load_dotenv()
DATA, OUT = Path("data"), Path("out")
OUT.mkdir(exist_ok=True)
TASKS = ["topic", "entails", "positive", "similarity"]
COLOR = {"topic": "#2a78d6", "entails": "#eb6834", "positive": "#1baf7a", "similarity": "#eda100"}
pd.set_option("display.precision", 3)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
read_jsonl = lambda p: [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

assert (DATA / "eval_v1.jsonl").exists(), "평가셋이 없다. nb00_setup → nb08_evaluation 을 먼저 실행한다"

JEV_URL = os.getenv("JEV_URL", "https://api.typesafe.ai")
http = requests.Session()
http.mount(JEV_URL, HTTPAdapter(max_retries=Retry(total=4, backoff_factor=0.5, status_forcelist=[408, 429, 500, 502, 503, 504])))

def ask(state, questions, model="jev-latest"):
    r = http.post(f"{JEV_URL}/v1/systemone", timeout=60, headers={"Authorization": f"Bearer {os.environ['TYPESAFE_API_KEY']}"},
                  json={"model": model, "state": state, "questions": questions})
    r.raise_for_status()
    return r.json() | {"_latency_s": r.elapsed.total_seconds()}

choice = lambda instructions, criteria: {"type": "choice", "instructions": instructions, "criteria": criteria}
score = lambda instructions, levels: {"type": "score", "instructions": instructions, "criteria": levels}
noul = lambda instructions, true="", false="": {"type": "noul", "instructions": instructions} | ({"criteria": {"true": true, "false": false}} if true else {})

def probs_of(answer):
    return answer.get("probabilities") or {"true": answer["noul"], "false": 1 - answer["noul"]}

def judge(item, resp):
    (task, q), = item["questions"].items()
    a = resp["answers"][task]
    pr = {str(k): float(v) for k, v in probs_of(a).items()}
    best = max(pr, key=pr.get)
    pred = {"noul": lambda: pr["true"] >= 0.5, "choice": lambda: best, "score": lambda: int(best)}[q["type"]]()
    return {"id": item["id"], "task": item["task"], "qtype": q["type"], "gold": item["gold"], "model": resp["model"],
            "pred": pred, "probs": pr, "conf": pr[best], "conf_field": a.get("confidence"), "correct": bool(pred == item["gold"]),
            "input_tokens": resp["usage"]["input_tokens"], "latency_s": resp["_latency_s"]}

def run(items, engine=lambda it: ask(it["state"], it["questions"]), workers=4):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(lambda it: judge(it, engine(it)), items))

arrays = lambda recs: (np.array([r["conf"] for r in recs]), np.array([r["correct"] for r in recs], float))

def ece(conf, ok, bins=15):
    idx = np.clip(np.searchsorted(np.linspace(0, 1, bins + 1), conf, side="left") - 1, 0, bins - 1)
    return float(sum(abs(conf[idx == i].mean() - ok[idx == i].mean()) * (idx == i).sum() for i in np.unique(idx)) / len(conf))

import csv
from collections import Counter

RAW = DATA / "raw"
EVAL_Q = {i["task"]: i["questions"] for i in read_jsonl(DATA / "eval_v1.jsonl")}      # 평가셋과 같은 질문을 쓴다
TOPICS = list(EVAL_Q["topic"]["topic"]["criteria"])
load_json = lambda name: json.loads((RAW / name).read_text(encoding="utf-8"))

def load_nsmc(name):
    rows = list(csv.DictReader((RAW / name).open(encoding="utf-8"), delimiter="\t", quoting=csv.QUOTE_NONE))
    return [r for r in rows if r["document"] and r["document"].strip()]

def item_topic(r):
    votes = Counter(r["annotations"]["annotations"]["first-scope"])
    return {"id": r["guid"], "task": "topic", "state": r["title"], "gold": r["label"], "gold_dist": {k: votes.get(k, 0) / sum(votes.values()) for k in TOPICS}}

def item_entails(r):
    votes = [r[k] for k in ("author", "label2", "label3", "label4", "label5")]
    return {"id": r["guid"], "task": "entails", "state": {"premise": r["premise"], "hypothesis": r["hypothesis"]},
            "gold": r["gold_label"] == "entailment", "gold_dist": {"true": votes.count("entailment") / len(votes)}}

def item_positive(r):
    return {"id": f"nsmc_{r['id']}", "task": "positive", "state": r["document"].strip(), "gold": r["label"] == "1", "gold_dist": None}

def item_similarity(r):
    c = Counter(int(round(a)) for a in r["annotations"]["annotations"])
    return {"id": r["guid"], "task": "similarity", "state": {"sentence1": r["sentence1"], "sentence2": r["sentence2"]},
            "gold": int(round(r["labels"]["label"])), "gold_value": r["labels"]["label"], "gold_dist": {str(k): c.get(k, 0) / sum(c.values()) for k in range(6)}}

TRAIN_SOURCES = {"topic": ("ynat-v1.1_train.json", load_json, item_topic), "entails": ("klue-nli-v1.1_train.json", load_json, item_entails),
                 "positive": ("ratings_train.txt", load_nsmc, item_positive), "similarity": ("klue-sts-v1.1_train.json", load_json, item_similarity)}
load_train = lambda task: [TRAIN_SOURCES[task][2](r) for r in TRAIN_SOURCES[task][1](TRAIN_SOURCES[task][0])]

TRAIN, TEACHER = DATA / "train", Path("results/teacher")
TRAIN.mkdir(parents=True, exist_ok=True); TEACHER.mkdir(parents=True, exist_ok=True)
write_jsonl = lambda p, rows: Path(p).write_text("".join(json.dumps(r, ensure_ascii=False, default=str) + "\n" for r in rows), encoding="utf-8")
N_PER_TASK = 1500                       # 6,000 = Laya typed-decisions 체크포인트의 학습 데이터 규모

## 13.1 학습 데이터 샘플링

평가셋은 dev 분할에서 뽑았다(8장). 학습 데이터는 train 분할에서만 뽑으므로 평가셋과 겹치지 않는다. 과제마다 시드를 고정해 1,500개를 뽑고, 항목마다 평가셋과 같은 질문을 붙인다.

In [2]:
train = {t: [i | {"questions": EVAL_Q[t]} for i in random.Random(2026).sample(load_train(t), N_PER_TASK)] for t in TASKS}
eval_ids = {i["id"] for i in read_jsonl(DATA / "eval_v1.jsonl")}
print("평가셋과 겹치는 항목:", sum(i["id"] in eval_ids for rows in train.values() for i in rows))
pd.DataFrame({t: pd.Series([str(i["gold"]) for i in rows]).value_counts(normalize=True).round(3).to_dict() for t, rows in train.items()}).fillna("")

평가셋과 겹치는 항목: 0


,topic,entails,positive,similarity
세계,0.183,,,
정치,0.173,,,
스포츠,0.163,,,
경제,0.141,,,
IT과학,0.119,,,
생활문화,0.117,,,
사회,0.104,,,
False,,0.66,0.501,
True,,0.34,0.499,
4,,,,0.355


## 13.2 주석자 분포

KLUE 는 여러 사람이 매긴 표를 남겼다. 표가 한쪽으로 몰리지 않은 항목은 정답이 하나가 아니다. 그런 항목에서 정답 하나만 가르치면, 모델은 사람도 헷갈린 문제에 확신하도록 배운다.

교사 분포의 키는 질문 유형마다 정해져 있다. choice 는 선택지 이름, score 는 "0"…"5", noul 은 "false"·"true" 다.

In [3]:
KEYS = {t: {"choice": lambda q: list(q["criteria"]), "score": lambda q: [str(i) for i in range(len(q["criteria"]))],
            "noul": lambda q: ["false", "true"]}[q["type"]](q) for t, qs in EVAL_Q.items() for q in qs.values()}
gold_key = lambda i: str(i["gold"]).lower() if i["task"] in ("entails", "positive") else str(i["gold"])

def hard(i, smoothing=0.1):
    return {k: (1 - smoothing) * (k == gold_key(i)) + smoothing / len(KEYS[i["task"]]) for k in KEYS[i["task"]]}

HUMAN = {"topic": lambda i: {k: i["gold_dist"][k] for k in KEYS["topic"]},
         "entails": lambda i: {"false": 1 - i["gold_dist"]["true"], "true": i["gold_dist"]["true"]},
         "positive": lambda i: hard(i, 0.0),                                   # NSMC 는 표가 없다
         "similarity": lambda i: {k: i["gold_dist"][k] for k in KEYS["similarity"]}}
human = lambda i: HUMAN[i["task"]](i)
norm_entropy = lambda d: float(-sum(p * np.log(p) for p in d.values() if p > 0) / np.log(len(d)))

pd.DataFrame({t: {"표가 갈린 항목": np.mean([max(human(i).values()) < 1 for i in rows]),
                  "다수 표의 평균 비율": np.mean([max(human(i).values()) for i in rows]),
                  "정규화 엔트로피 평균": np.mean([norm_entropy(human(i)) for i in rows])} for t, rows in train.items()}).T

,표가 갈린 항목,다수 표의 평균 비율,정규화 엔트로피 평균
topic,0.330,0.890,0.108
entails,0.110,0.973,0.086
positive,0.000,1.000,0.000
similarity,0.805,0.668,0.383


## 13.3 Jev 교사

학습 항목 6,000개를 8장의 채점 함수로 Jev 에 보낸다. 기록에는 확률 분포 전체가 남으므로 그대로 교사 분포가 된다. 학습 데이터에도 정답이 있으니 교사가 얼마나 맞히는지 함께 잰다.

In [5]:
summary = lambda rs: {"n": len(rs), "정확도": arrays(rs)[1].mean(), "평균 확신": arrays(rs)[0].mean(), "ECE": ece(*arrays(rs)),
                      "입력 토큰": sum(r["input_tokens"] for r in rs)}
t0 = time.perf_counter()
jev_rec = {t: run(rows) for t, rows in train.items()}
write_jsonl(TEACHER / "jev_train.jsonl", [r for rs in jev_rec.values() for r in rs])
print(f"{time.perf_counter() - t0:.0f}초")
pd.DataFrame({t: summary(rs) for t, rs in jev_rec.items()}).T.assign(비용_달러=lambda d: d["입력 토큰"] * 0.042 / 1e6)

270초


,n,정확도,평균 확신,ECE,입력 토큰,비용_달러
topic,1500.0,0.799,0.887,0.089,758653.0,0.032
entails,1500.0,0.957,0.919,0.045,619802.0,0.026
positive,1500.0,0.863,0.831,0.032,541888.0,0.023
similarity,1500.0,0.435,0.669,0.235,688195.0,0.029


## 13.4 로컬 LLM 교사

같은 6,000개를 로컬 LLM 에 묻는다. 생성한 글이 아니라 첫 토큰의 확률을 읽는다(AnyJev 방식, 9장). 선택지 순서를 한 칸씩 돌린 프롬프트를 한 배치로 보내 위치 편향을 상쇄한다. 생성기로 쓰면 답 하나와 스스로 적은 확신밖에 얻지 못하지만, 이렇게 읽으면 선택지마다 확률이 나와 교사 분포가 된다.

GPU 에서 수십 분 걸린다. 끝나면 파일로 남기고 모델을 내린다. 14장의 학습은 이 파일만 읽는다.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = os.environ["LOCAL_LLM_MODEL"]
tok = AutoTokenizer.from_pretrained(LLM_MODEL, padding_side="left")
tok.pad_token = tok.pad_token or tok.eos_token
lm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, dtype="auto", device_map="auto")

def chat_batch(prompts, max_new_tokens=256, batch_size=16, system=None):
    # 프롬프트 여럿을 묶어 generate 한 번에 보낸다(왼쪽 패딩). 서빙 엔진 없이 처리량을 내는 방법이다
    msgs = lambda p: [{"role": "system", "content": system}] * bool(system) + [{"role": "user", "content": p}]
    texts = [tok.apply_chat_template(msgs(p), tokenize=False, add_generation_prompt=True, enable_thinking=False) for p in prompts]
    outs = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], return_tensors="pt", padding=True).to(lm.device)
        gen = lm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

chat = lambda prompt, **kw: chat_batch([prompt], **kw)[0]
print(LLM_MODEL, "|", lm.device)

import torch

LETTERS = "ABCDEFGHIJ"
letter_ids = [tok.encode(c, add_special_tokens=False)[0] for c in LETTERS]
OPTIONS = {"choice": lambda c: [(k, f"{k}: {v}") for k, v in c.items()],
           "score": lambda c: [(str(i), lv) for i, lv in enumerate(c)],
           "noul": lambda c: [("true", f"참. {c.get('true', '')}"), ("false", f"거짓. {c.get('false', '')}")]}

def anyjev_prompt(state, q, texts, order):
    menu = "\n".join(f"{LETTERS[s]}. {texts[j]}" for s, j in enumerate(order))
    user = f"[판단할 대상]\n{json.dumps(state, ensure_ascii=False)}\n\n[질문]\n{q['instructions']}\n{menu}\n\n답(기호 한 글자):"
    msgs = [{"role": "system", "content": "너는 판단만 하는 분류기다. 질문을 읽고 선택지 기호 한 글자로만 답한다."}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def rotations(state, q, cyclic=True):
    # 선택지 순서를 한 칸씩 돌린 k 개 프롬프트를 한 배치로 보내고, 첫 토큰에서 기호들의 로그 확률을 읽는다
    keys, texts = zip(*OPTIONS[q["type"]](q.get("criteria") or {}))
    k = len(keys)
    orders = np.array([[(i + s) % k for i in range(k)] for s in range(k if cyclic else 1)])
    enc = tok([anyjev_prompt(state, q, texts, o) for o in orders], return_tensors="pt", padding=True).to(lm.device)
    with torch.no_grad():
        logits = lm(**enc).logits[:, -1].float()
    slot = torch.log_softmax(logits, -1)[:, letter_ids[:k]].cpu().numpy()       # [회전, 자리]
    R = np.zeros_like(slot)
    np.put_along_axis(R, orders, slot, axis=1)                                  # 자리 → 원래 선택지 순서
    return list(keys), R, int(enc["attention_mask"].sum())

ANSWER = {"noul": lambda keys, p: {"type": "noul", "noul": float(p[keys.index("true")])},
          "choice": lambda keys, p: {"type": "choice", "choice": keys[int(p.argmax())], "probabilities": dict(zip(keys, map(float, p)))},
          "score": lambda keys, p: {"type": "score", "score": float(np.dot(np.arange(len(p)), p)), "probabilities": dict(zip(keys, map(float, p)))}}

def anyjev(state, questions):
    t0, answers, tokens = time.perf_counter(), {}, 0
    for name, q in questions.items():
        keys, R, n = rotations(state, q)
        p = np.exp(R.mean(0) - R.mean(0).max())                                 # 회전 평균 로그 확률 → 확률
        answers[name], tokens = ANSWER[q["type"]](keys, p / p.sum()), tokens + n
    return {"model": "anyjev:" + LLM_MODEL, "answers": answers, "usage": {"input_tokens": tokens}, "_latency_s": time.perf_counter() - t0}

t0 = time.perf_counter()
llm_rec = {t: run(rows, engine=lambda it: anyjev(it["state"], it["questions"]), workers=1) for t, rows in train.items()}
write_jsonl(TEACHER / "llm_train.jsonl", [r for rs in llm_rec.values() for r in rs])
print(f"{time.perf_counter() - t0:.0f}초")
del lm; torch.cuda.empty_cache()
pd.DataFrame({t: summary(rs) for t, rs in llm_rec.items()}).T.drop(columns="입력 토큰")

/home/josh/anaconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignoring corrupted tree cache file /data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json: [Errno 13] Permission denied: '/data/shared/hf/hub/models--Qwen--Qwen3.5-4B/trees/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a.json'
Loading weights: 100%|██████████| 426/426 [00:01<00:00, 377.25it/s]


Qwen/Qwen3.5-4B | cuda:0


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


529초


,n,정확도,평균 확신,ECE
topic,1500.0,0.767,0.847,0.089
entails,1500.0,0.929,0.870,0.058
positive,1500.0,0.815,0.874,0.070
similarity,1500.0,0.485,0.550,0.088


## 13.5 두 교사가 갈린 항목

Jev 와 로컬 LLM 이 다른 답을 고른 항목을 센다. 갈린 항목에서는 누가 맞았는지, 갈린 항목의 확신은 어떤지 본다. 교사 둘이 갈린 항목은 사람 표도 갈려 있을 가능성이 높다. 이런 항목을 따로 표시해 두면, 학습에서 빼거나 사람에게 다시 맡길 후보가 된다.

In [7]:
rows = []
for t in TASKS:
    for j, l, i in zip(jev_rec[t], llm_rec[t], train[t]):
        rows.append({"과제": t, "갈림": j["pred"] != l["pred"], "Jev 맞음": j["correct"], "LLM 맞음": l["correct"],
                     "Jev 확신": j["conf"], "LLM 확신": l["conf"], "사람 표 갈림": max(human(i).values()) < 1})
D = pd.DataFrame(rows)
pd.concat({"전체": D.groupby("과제")[["갈림", "사람 표 갈림"]].mean(),
           "갈린 항목에서": D[D["갈림"]].groupby("과제")[["Jev 맞음", "LLM 맞음", "Jev 확신", "LLM 확신", "사람 표 갈림"]].mean()}, axis=1)

전체         갈린 항목에서                             
               갈림 사람 표 갈림  Jev 맞음 LLM 맞음 Jev 확신 LLM 확신 사람 표 갈림
과제                                                            
entails     0.057   0.110   0.753  0.247  0.773  0.661   0.365
positive    0.135   0.000   0.678  0.322  0.673  0.720   0.000
similarity  0.438   0.805   0.288  0.402  0.592  0.472   0.896
topic       0.165   0.330   0.528  0.331  0.733  0.647   0.460

## 13.6 교사 품질 평가

교사가 학생의 천장이다. 평가셋에서 교사 분포 자체의 품질을 잰다. 사람 표는 평가셋 항목에도 있다. Jev 는 8장, 로컬 LLM(AnyJev)은 12장에서 평가셋 기록을 남겼다.

- argmax 정확도: 분포에서 가장 큰 선택지가 정답과 같은 비율
- soft 정확도: 정답 선택지에 준 확률의 평균
- 사람과의 거리(TV): 교사 분포와 사람 표 분포의 총변동 거리. 0 이면 같다

In [8]:
items = read_jsonl(DATA / "eval_v1.jsonl")
rec_of = lambda name: {r["id"]: r["probs"] for r in read_jsonl(f"results/eval_v1/{name}.records.jsonl")}
jev_eval, llm_eval = rec_of("jev"), rec_of("anyjev")
mix = lambda a, b_, w=0.5: {k: w * a[k] + (1 - w) * b_[k] for k in a}
tv = lambda a, b_: 0.5 * sum(abs(a[k] - b_[k]) for k in a)
TEACHERS = {"hard": hard, "human": human, "jev": lambda i: {k: jev_eval[i["id"]][k] for k in KEYS[i["task"]]},
            "llm": lambda i: {k: llm_eval[i["id"]][k] for k in KEYS[i["task"]]},
            "mix": lambda i: mix({k: jev_eval[i["id"]][k] for k in KEYS[i["task"]]}, human(i))}
pd.DataFrame([{"과제": t, "교사": name, "argmax 정확도": np.mean([max(d := f(i), key=d.get) == gold_key(i) for i in its]),
               "soft 정확도": np.mean([f(i)[gold_key(i)] for i in its]), "사람과의 거리(TV)": np.mean([tv(f(i), human(i)) for i in its])}
              for t in TASKS for its in [[i for i in items if i["task"] == t]] for name, f in TEACHERS.items()]).set_index(["과제", "교사"])

argmax 정확도  soft 정확도  사람과의 거리(TV)
과제         교사                                      
topic      hard        1.000     0.914        0.174
           human       1.000     0.873        0.000
           jev         0.800     0.772        0.260
           llm         0.765     0.736        0.274
           mix         0.915     0.823        0.130
entails    hard        1.000     0.950        0.066
           human       1.000     0.973        0.000
           jev         0.975     0.912        0.084
           llm         0.935     0.849        0.142
           mix         1.000     0.942        0.042
positive   hard        1.000     0.950        0.050
           human       1.000     1.000        0.000
           jev         0.850     0.778        0.222
           llm         0.790     0.764        0.236
           mix         1.000     0.889        0.111
similarity hard        1.000     0.917        0.441
           human       0.705     0.533        0.000
           jev         0.350     0.344        0.437
           llm         0.250     0.261        0.499
           mix         0.590     0.439        0.218

## 13.7 학습 파일과 dry_run

교사마다 Laya 학습 형식의 JSONL 을 쓴다. 한 줄에 state, 질문, 질문마다의 교사 분포가 들어간다. 파일마다 SHA-256 을 카드에 적어, 어느 교사로 학습한 체크포인트인지 16장에서 추적한다.

In [9]:
import hashlib

jev_train = {r["id"]: r["probs"] for r in read_jsonl(TEACHER / "jev_train.jsonl")}
llm_train = {r["id"]: r["probs"] for r in read_jsonl(TEACHER / "llm_train.jsonl")}
pick = lambda probs, i: {k: probs[i["id"]][k] for k in KEYS[i["task"]]}
BUILD = {"hard": hard, "human": human, "jev": lambda i: pick(jev_train, i), "llm": lambda i: pick(llm_train, i),
         "mix": lambda i: mix(pick(jev_train, i), human(i))}
laya_row = lambda i, dist: {"id": i["id"], "task": i["task"], "state": i["state"], "questions": i["questions"],
                            "gold": {i["task"]: {"probabilities": {k: round(float(dist[k]), 6) for k in KEYS[i["task"]]}}}}
card = {}
for name, f in BUILD.items():
    write_jsonl(TRAIN / f"{name}.jsonl", [laya_row(i, f(i)) for t in TASKS for i in train[t]])
    card[name] = {"rows": 4 * N_PER_TASK, "sha256": hashlib.sha256((TRAIN / f"{name}.jsonl").read_bytes()).hexdigest()}
(TRAIN / "card.json").write_text(json.dumps({"n_per_task": N_PER_TASK, "seed": 2026, "teachers": card}, indent=1))
pd.DataFrame(card).T.assign(sha256=lambda d: d["sha256"].str[:16])

,rows,sha256
hard,6000,42b7870efc584fc1
human,6000,e3c333b3fff94473
jev,6000,9161fa997d08c070
llm,6000,3d06ce3f837962fc
mix,6000,ee2416e6d55d519f


`laya.train.dry_run` 은 가중치를 올리지 않고 토크나이저로 항목을 만들어 본다. 건너뛴 질문과 이유, 학습이 쓸 optimizer 갱신 횟수를 알려 준다.

In [10]:
from laya.train import TrainConfig, dry_run
s = dry_run(str(TRAIN / "human.jsonl"), "multilingual", TrainConfig())
{k: s[k] for k in ["rows_read", "valid_items", "train_items", "calibration_items", "effective_batch", "optimizer_updates", "skipped", "max_len", "head_max_len"]}

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1606.15it/s]


dry-run: 6000 rows read, 6000 valid items, skipped: {}
dry-run: optimizer budget: 5600 train item(s), effective batch up to 64 (micro_batch 8 x grad_accum 8), 352 optimizer updates over 4 epoch(s)


{'rows_read': 6000,
 'valid_items': 6000,
 'train_items': 5600,
 'calibration_items': 400,
 'effective_batch': 64,
 'optimizer_updates': 352,
 'skipped': {},
 'max_len': 1024,
 'head_max_len': 256}

## 정리

이 노트북이 잰 것과 만든 것이다. 결과 해석은 13장 본문에서 한다.

- 13.1 과제마다 1,500개의 학습 항목, 평가셋과 겹침 0
- 13.2 사람 표가 갈린 항목의 비율과 엔트로피
- 13.3·13.4 학습 데이터에 대한 Jev 교사와 로컬 LLM 교사의 정확도·ECE (`results/teacher/`)
- 13.5 두 교사가 갈린 항목과 그 항목에서의 정답률
- 13.6 평가셋에서 교사 다섯의 argmax·soft 정확도와 사람과의 거리
- 13.7 `data/train/{hard,human,jev,llm,mix}.jsonl` 과 카드, dry_run 의 건너뜀·갱신 횟수